In [ ]:
import sys
import os
import random
import importlib

In [ ]:
sys.path.append('/home/eraslab1/Projects/PerturbDecodeMulti')

In [ ]:
from Src import pertdec
from Src.utils.libraries import *

In [ ]:
# adata = sc.read("/home/eraslab1/Projects/AbbasScreen/Data/adataALL.h5ad")
# adata = adata[(adata.obs["guide_num"] == 1) | (adata.obs["guide_num"] ==100),]
# adata.obs["full_guide_id"] = adata.obs["full_guide_id"].astype(str)
# adata.obs.loc[adata.obs["Sample_type"]=="adenocarcinoma", "full_guide_id"] = "adenocarcinoma"
# adata.obs.loc[adata.obs["Sample_type"]=="NEPC", "full_guide_id"] = "NEPC"
# del adata.raw
# adata.write("/home/eraslab1/Projects/AbbasScreen/Data/adataSingles.h5ad")

In [ ]:
adata = sc.read("/home/eraslab1/Projects/AbbasScreen/Data/adataSingles.h5ad")

In [ ]:
adata.X

In [ ]:
guideCounts = pd.DataFrame(adata.obs['full_guide_id'].value_counts())

In [ ]:
guideCounts.to_csv("/home/eraslab1/Projects/AbbasScreen/TextFiles/GuideCounts.csv")

In [ ]:
guideCounts = pd.read_csv("/home/eraslab1/Projects/AbbasScreen/TextFiles/GuideCounts.csv", index_col=0)
guides=list(guideCounts.index)
guides.sort()
guides = ['NTC_108'] + guides[0:guides.index('NTC_108')] + guides[guides.index('NTC_108')+1:]

In [ ]:
# pertdec.createTrainValData(inAdata=adata,
#                            perturbationColumn='full_guide_id',
#                            pertCategories= guides,
#                            dataDir='/home/eraslab1/Projects/AbbasScreen/Data/',
#                            valSetPercent = 0.2)

In [ ]:
# pertdec.runTrainingComBVAE(  model_dir="/home/eraslab1/Projects/AbbasScreen/COMBVAEModel/",
#                              trainfile="/home/eraslab1/Projects/AbbasScreen/Data/pertDecTrain.h5ad",
#                              valfile="/home/eraslab1/Projects/AbbasScreen/Data/pertDecValidation.h5ad",
#                              perturbationColumn="full_guide_id",
#                              pertCategories=guides,
#                              n_inputs=18499,
#                              n_cond_in=7085, 
#                              n_latents = 128, 
#                              n_cond = 128,
#                              model_type='CVAE_basic',
#                              beta=6.0,
#                              batch_size=100,
#                              lr=0.0001,
#                              weight_decay=1e-5,
#                              use_gpu=True, setGPU=False,
#                              optimizer='adam', max_epochs=3, 
#                              scheduler='none')
    

In [ ]:
perturbations, factoredOutCellEmbeddings, perturbationEmbeddings, perturbationsList = pertdec.extract_model_embeddings(
                                            model_dir="/home/eraslab1/Projects/AbbasScreen/COMBVAEModel/", 
                                            datafile="/home/eraslab1/Projects/AbbasScreen/Data/adataSingles.h5ad", 
                                            perturbationColumn="full_guide_id", 
                                            pertCategories=guides, 
                                            n_inputs=18499, 
                                            n_cond_in=7085, 
                                            n_latents=128, 
                                            n_cond=128, 
                                            batch_size = 126,
                                            model_type='CVAE_basic')

In [ ]:
type(perturbationEmbeddings)

In [ ]:
type(perturbationsList)

In [ ]:
factoredOutCellEmbeddings.shape

In [ ]:
len(perturbationsList)

In [ ]:
# with open("perturbations.pkl", "wb") as f:
#     pickle.dump(perturbations, f)
# with open("factoredOutCellEmbeddings.pkl", "wb") as f:
#     pickle.dump(factoredOutCellEmbeddings, f)
# with open("perturbationEmbeddings.pkl", "wb") as f:
#     pickle.dump(perturbationEmbeddings, f)
# with open("perturbationsList.pkl", "wb") as f:
#     pickle.dump(perturbationsList, f)

In [ ]:
with open("perturbations.pkl", "rb") as f:
    perturbations = pickle.load(f)
with open("factoredOutCellEmbeddings.pkl", "rb") as f:
    factoredOutCellEmbeddings = pickle.load(f)
with open("perturbationEmbeddings.pkl", "rb") as f:
    perturbationEmbeddings = pickle.load(f)
with open("perturbationsList.pkl", "rb") as f:
    perturbationsList = pickle.load(f)


In [ ]:
perturbationsList

In [ ]:
pertEmbedAnndat, visAnnDat = pertdec.visualizePerturbationEmbeddings(perturbationEmbeddings, 
                                                          perturbationsList,
                                                          clusteringRes=1.5)

In [ ]:
visAnnDat

In [ ]:
selectedGuides, df = pertdec.selectWorkingGuides(pertEmbedAnndat, 
                                                 controlGuideIdentifiers=["NTC"],
                                                 numberOfGuidesPerTarget=4,
                                                 pValThreshold=0.05,
                                                 correlationThreshold = 0,
                                                 method='pearson')

In [ ]:
pd.DataFrame(selectedGuides).to_csv("SelectedGuides.csv", index=False)

In [ ]:
controlGuides = list(adata.obs.loc[[x.split("_")[0] in "NTC" for x in adata.obs["full_guide_id"]],"full_guide_id"].unique())
controlGuides.sort()

In [ ]:
inGuides = selectedGuides + controlGuides + ["adenocarcinoma", "NEPC"]

In [ ]:
adata_selected = adata[[x in inGuides for x in adata.obs["full_guide_id"]]]

In [ ]:
adata_selected.obs["SelectedPerturbations"] = [x.split("_")[0]  for x in adata_selected.obs["full_guide_id"]]

In [ ]:
adata_selected.obs["SelectedPerturbations"]

In [ ]:
pertdec.inferEffectSizes(adata=adata_selected,
                         perturbationsColumn="SelectedPerturbations",
                         referenceLevel="NTC",
                         covariates=["n_genes", "mt_frac", "log10_n_umis" ],
                         par_test_target_interval = 250)

In [ ]:
effectSizes = pd.DataFrame(pd.read_csv("./TmpOLSOuts/EffectSizeEstimates.csv"))

In [ ]:
effectSizes = effectSizes.drop_duplicates(subset=['dependent_variable', 'independent_variable'])

In [ ]:
coefs_df = effectSizes.pivot(index='dependent_variable', 
                            columns='independent_variable',
                            values='coefficient')
pvals_df = effectSizes.pivot(index='dependent_variable', 
                            columns='independent_variable',
                            values='p_value')

In [ ]:
coefs_df

In [ ]:
coefs_df.to_csv("./TmpOLSOuts/LogFCs.csv")

In [ ]:
pvals_df.to_csv("./TmpOLSOuts/Pvalues.csv")

In [ ]:
from statsmodels.stats.multitest import multipletests

#fdr_df = pvals_df.apply(lambda col: multipletests(col, method='fdr_bh')[1])
fdr_df = pvals_df.apply(lambda row: pd.Series(multipletests(row, method='fdr_bh')[1], index=row.index), axis=1)


In [ ]:
fdr_df.to_csv("./TmpOLSOuts/FDRs.csv")

In [ ]:
coefs_df_fdrCor = coefs_df.copy()

In [ ]:
coefs_df_fdrCor[fdr_df > 0.1] = 0 

In [ ]:
coefs_df_fdrCor.to_csv("LogFCs_FDR_01_zeroedOut.csv")

In [ ]:
coefs_df_fdrCor = coefs_df_fdrCor.drop(["n_genes","log10_n_umis","const","mt_frac"],axis=1)

In [ ]:
effectedGenes = pd.DataFrame((fdr_df < 0.1).sum(axis=1))

In [ ]:
effectedGenes.iloc[:,0] > 5

In [ ]:
coefs_df_fdrCor_red = coefs_df_fdrCor.loc[effectedGenes.iloc[:,0] > 5,:]

In [ ]:
coefs_df_fdrCor_red

In [ ]:
adenoDist = coefs_df_fdrCor_red.subtract(coefs_df_fdrCor_red['adenocarcinoma'], axis=0)

In [ ]:
adenoDist

In [ ]:
l1_norms = adenoDist.apply(lambda col: np.linalg.norm(col, ord=1))

In [ ]:
l1_norms = l1_norms.sort_values()

In [ ]:
pd.DataFrame(l1_norms).to_csv("L1_distance_to_adenocarcinoma.csv")

In [ ]:
effectivePerturbations = pd.DataFrame((fdr_df < 0.1).sum(axis=0))

In [ ]:
effectivePerturbations

In [ ]:
fdr_df = fdr_df.drop(["n_genes","log10_n_umis","const","mt_frac"],axis=1)
effectedGenes = pd.DataFrame((fdr_df < 0.1).sum(axis=1))

In [ ]:
effectedGenes.loc[effectedGenes.iloc[:,0] >0,]

In [ ]:
myRes = pd.DataFrame((fdr_df < 0.1).sum())

In [ ]:
myRes

In [ ]:
# Plot histogram
myRes.iloc[:,0].plot.hist(bins=2000, edgecolor='black')

plt.xlim((0,100))
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of number of DE genes per perturbation')
plt.show()